In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/ThesisExperiment_cleann/detector_comparison'
os.chdir(OUT_DIR)
print(os.listdir(OUT_DIR))
!pip install -q pandas scipy
!python verify_detector_correlations.py

In [ ]:
"""
verify_detector_correlations.py

Reproduces the Detectorv1-vs-Detectorv2 AU4/5/7/9 correlations cited in the
thesis (Methodology, "Tool Selection: Detectorv1 vs. Detectorv2", and
Limitations).

Data: Detectorv1 and Detectorv2 were run on the same clip (participant 1,
Robot1 trial, converted to temp_comparison_test.mp4, every 5th frame,
416 frames) in PyFeat_Comparison_Colab.ipynb. Two independently saved pairs
exist (Pair A, Pair B).

INPUT (same folder as this script):
    comparison_v1_output.csv    comparison_v2_output.csv      (Pair A)
    comparison_v1_output-2.csv  comparison_v2_output-2.csv    (Pair B)

OUTPUT (written to the folder "detector_comparison_results"):
    detector_correlations.csv   one row per pair x AU: n frames, Pearson r, p,
                                95% CI, mean AU value of each detector
    detector_correlations_log.txt   everything printed below, plus date,
                                package versions and a fingerprint (SHA-256)
                                of each input file
"""

import os
import sys
import hashlib
import datetime
import numpy as np
import pandas as pd
import scipy
from scipy.stats import pearsonr

AUS = ["AU04", "AU05", "AU07", "AU09"]
PAIRS = [
    ("Pair A", "comparison_v1_output.csv", "comparison_v2_output.csv"),
    ("Pair B", "comparison_v1_output-2.csv", "comparison_v2_output-2.csv"),
]
OUT_DIR = "detector_comparison_results"
os.makedirs(OUT_DIR, exist_ok=True)

log_lines = []
def say(text=""):
    print(text)
    log_lines.append(text)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

say("Detectorv1 vs Detectorv2: frame-level correlations on the same clip")
say(f"Run on: {datetime.datetime.now():%Y-%m-%d %H:%M}")
say(f"Python {sys.version.split()[0]}, pandas {pd.__version__}, "
    f"numpy {np.__version__}, scipy {scipy.__version__}")
say()

rows = []
for label, v1_file, v2_file in PAIRS:
    if not (os.path.exists(v1_file) and os.path.exists(v2_file)):
        say(f"--- {label}: skipped (file not found: {v1_file} / {v2_file}) ---")
        say()
        continue
    v1 = pd.read_csv(v1_file).sort_values("frame").reset_index(drop=True)
    v2 = pd.read_csv(v2_file).sort_values("frame").reset_index(drop=True)
    merged = pd.merge(v1[["frame"] + AUS], v2[["frame"] + AUS],
                      on="frame", suffixes=("_v1", "_v2"))
    n = len(merged)

    say(f"--- {label}: {v1_file} vs {v2_file} (n={n} matched frames) ---")
    say(f"    SHA-256 {v1_file}: {sha256(v1_file)}")
    say(f"    SHA-256 {v2_file}: {sha256(v2_file)}")
    for au in AUS:
        a, b = merged[f"{au}_v1"], merged[f"{au}_v2"]
        r, p = pearsonr(a, b)
        z, se = np.arctanh(r), 1 / np.sqrt(n - 3)
        lo, hi = np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)
        say(f"  {au}: r = {r:.2f}  (95% CI {lo:.2f} to {hi:.2f}, p = {p:.4g}); "
            f"mean v1 = {a.mean():.3f}, mean v2 = {b.mean():.3f}")
        rows.append({"pair": label, "AU": au, "n_frames": n,
                     "pearson_r": round(r, 4), "p_value": p,
                     "ci95_low": round(lo, 4), "ci95_high": round(hi, 4),
                     "mean_detectorv1": round(a.mean(), 4),
                     "mean_detectorv2": round(b.mean(), 4)})
    say()

pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "detector_correlations.csv"), index=False)
with open(os.path.join(OUT_DIR, "detector_correlations_log.txt"), "w", encoding="utf8") as f:
    f.write("\n".join(log_lines) + "\n")
print(f"Saved results to the folder '{OUT_DIR}':")
print("  detector_correlations.csv")
print("  detector_correlations_log.txt")